In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


In [2]:
df = pd.read_csv('dataset.csv')

In [3]:
DROP_COLS = [
    'comment', 'map_file', 'meta_data', 'direction', 'route_path',
    'citizen_accident_id', 'assigned_to_police_id',
    'age_of_truck', 'reason_breakdown', 'cargo_material',
    'resolved_at_address', 'resolved_at_latitude', 'resolved_at_longitude',
    'resolved_by_id', 'resolved_datetime',
    'description', 'kgid', 'veh_no',
    'id', 'address', 'end_address',  # free text / ID
    'map_file', 'last_modified_by_id', 'created_by_id',
    'modified_datetime', 'created_date',  # leakage risk / not features
    'closed_by_id', 'police_station', 'client_id', 'authenticated'
]
df_clean = df.drop(columns=[c for c in DROP_COLS if c in df.columns])

In [4]:
df_clean['closed_datetime']   = pd.to_datetime(df_clean['closed_datetime'],   utc=True, errors='coerce')
df_clean['start_datetime']    = pd.to_datetime(df_clean['start_datetime'],     utc=True, errors='coerce')
df_clean['resolved_datetime_fallback'] = pd.to_datetime(
    df.get('resolved_datetime', pd.Series()), utc=True, errors='coerce'
)

# Use closed_datetime first, fall back to resolved_datetime
df_clean['end_ts'] = df_clean['closed_datetime'].fillna(df_clean['resolved_datetime_fallback'])
df_clean['duration_mins'] = (
    df_clean['end_ts'] - df_clean['start_datetime']
).dt.total_seconds() / 60

In [5]:
df_clean['severity'] = pd.cut(
    df_clean['duration_mins'],
    bins=[0, 30, 90, 240, 1440],
    labels=['low', 'medium', 'high', 'critical']
)

In [6]:
df_clean['hour']        = df_clean['start_datetime'].dt.hour
df_clean['day_of_week'] = df_clean['start_datetime'].dt.dayofweek
df_clean['month']       = df_clean['start_datetime'].dt.month
df_clean['is_weekend']  = df_clean['day_of_week'].isin([5, 6]).astype(int)
df_clean['is_peak']     = df_clean['hour'].isin([7, 8, 9, 17, 18, 19]).astype(int)
df_clean['is_night']    = df_clean['hour'].isin(range(22, 24)).astype(int)

# ── 5. Geo features ───────────────────────────────────────────────────────────
df_clean['lat_bin'] = pd.cut(df_clean['latitude'],  bins=8, labels=False)
df_clean['lon_bin'] = pd.cut(df_clean['longitude'], bins=8, labels=False)
df_clean['has_end_location'] = (df_clean['endlatitude'].fillna(0) != 0).astype(int)


In [7]:
CAT_FILL_UNKNOWN = ['zone', 'junction', 'gba_identifier', 'veh_type', 'corridor']
for col in CAT_FILL_UNKNOWN:
    if col in df_clean.columns:
        df_clean[col] = df_clean[col].fillna('Unknown')

df_clean['priority'] = df_clean['priority'].fillna(df_clean['priority'].mode()[0])
df_clean['requires_road_closure'] = df_clean['requires_road_closure'].astype(int)

In [8]:
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split

In [9]:
LE_COLS = ['event_type', 'event_cause', 'veh_type', 'status']
le_encoders = {}
for col in LE_COLS:
    if col in df_clean.columns:
        le = LabelEncoder()
        df_clean[col + '_enc'] = le.fit_transform(df_clean[col].astype(str))
        le_encoders[col] = le

# High-cardinality: target encoding for zone, junction
# (avoids OHE explosion; robust for tree models)
target_col = 'duration_mins'
for col in ['zone', 'junction', 'corridor', 'gba_identifier']:
    if col in df_clean.columns:
        mean_map = df_clean.groupby(col)[target_col].mean()
        df_clean[col + '_tenc'] = df_clean[col].map(mean_map)
        # Add binary "is_known" flag alongside
        df_clean[col + '_known'] = (df_clean[col] != 'Unknown').astype(int)

# Priority ordinal encoding
priority_map = {'low': 0, 'medium': 1, 'high': 2, 'critical': 3}
if 'priority' in df_clean.columns:
    df_clean['priority_enc'] = df_clean['priority'].str.lower().map(priority_map)

In [10]:
FEATURE_COLS = [
    # temporal
    'hour', 'day_of_week', 'month', 'is_weekend', 'is_peak', 'is_night',
    # geo
    'latitude', 'longitude', 'lat_bin', 'lon_bin', 'has_end_location',
    # event attributes
    'event_type_enc', 'event_cause_enc', 'requires_road_closure',
    'veh_type_enc', 'status_enc',
    # encoded high-card
    'zone_tenc', 'zone_known',
    'junction_tenc', 'junction_known',
    'corridor_tenc', 'corridor_known',
    'gba_identifier_tenc', 'gba_identifier_known',
    # priority (when not the target)
    'priority_enc',
]
# Keep only columns that exist after all the steps above
FEATURE_COLS = [c for c in FEATURE_COLS if c in df_clean.columns]

In [11]:
from sklearn.model_selection import cross_val_score, StratifiedKFold, KFold
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    f1_score, roc_auc_score, classification_report, confusion_matrix
)
import xgboost as xgb
import lightgbm as lgb

In [12]:
X = df_clean[FEATURE_COLS].fillna(-999)   # -999 sentinel for trees
y_reg   = df_clean['duration_mins']
y_clf_b = df_clean['requires_road_closure']   # binary
y_clf_m = df_clean['severity'].cat.codes       # multi-class (0=low,1=med,2=high,3=critical)

X_train, X_test, y_reg_train, y_reg_test = train_test_split(
    X, y_reg, test_size=0.2, random_state=42
)
_, _, y_b_train, y_b_test = train_test_split(X, y_clf_b, test_size=0.2, random_state=42)
_, _, y_m_train, y_m_test = train_test_split(X, y_clf_m, test_size=0.2, random_state=42)

def regression_report(name, y_true, y_pred):
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2   = r2_score(y_true, y_pred)
    print(f"\n{'─'*40}")
    print(f"  {name}")
    print(f"  MAE : {mae:.2f} mins")
    print(f"  RMSE: {rmse:.2f} mins")
    print(f"  R²  : {r2:.4f}")
    return {'model': name, 'MAE': mae, 'RMSE': rmse, 'R2': r2}

def classification_report_short(name, y_true, y_pred, y_prob=None):
    f1 = f1_score(y_true, y_pred, average='weighted')
    auc = roc_auc_score(y_true, y_prob, multi_class='ovr', average='weighted') \
          if y_prob is not None and y_prob.ndim == 2 else None
    print(f"\n{'─'*40}")
    print(f"  {name}")
    print(f"  F1 (weighted): {f1:.4f}")
    if auc: print(f"  ROC-AUC:       {auc:.4f}")
    return {'model': name, 'F1': f1, 'AUC': auc}

In [16]:
# requires_road_closure is the TARGET for Task B
# It must NOT be in the feature set for Task B
# Check if it's in your FEATURE_COLS:
print("requires_road_closure in features:", 'requires_road_closure' in FEATURE_COLS)

# Fix: create a separate feature list for Task B
FEATURE_COLS_B = [c for c in FEATURE_COLS if c != 'requires_road_closure']

# Also audit: check correlation of all features with target
corr = X_train[FEATURE_COLS_B].corrwith(pd.Series(y_b_train.values, index=X_train.index))
print("\nTop correlations with road closure target:")
print(corr.abs().sort_values(ascending=False).head(10))

requires_road_closure in features: True

Top correlations with road closure target:
has_end_location    0.988061
month               0.261535
day_of_week         0.260427
hour                0.260078
event_type_enc      0.253718
veh_type_enc        0.135371
priority_enc        0.108919
event_cause_enc     0.076545
corridor_tenc       0.039427
is_weekend          0.033954
dtype: float64


In [14]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

In [17]:
X_tr_b = X_train[FEATURE_COLS_B]
X_te_b = X_test[FEATURE_COLS_B]

# ── Logistic Regression ───────────────────────────────────────────────────────
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

lr_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(class_weight='balanced', max_iter=500, random_state=42))
])
lr_pipe.fit(X_tr_b, y_b_train)
pred = lr_pipe.predict(X_te_b)
prob = lr_pipe.predict_proba(X_te_b)[:, 1]
print(classification_report(y_b_test, pred, target_names=['No closure', 'Closure']))
results_b = [{'model': 'LogisticReg',
              'F1': f1_score(y_b_test, pred, average='weighted'),
              'AUC': roc_auc_score(y_b_test, prob)}]

# ── Random Forest ─────────────────────────────────────────────────────────────
rf_clf = RandomForestClassifier(
    n_estimators=200, max_depth=8, class_weight='balanced',
    random_state=42, n_jobs=-1
)
rf_clf.fit(X_tr_b, y_b_train)
pred = rf_clf.predict(X_te_b)
prob = rf_clf.predict_proba(X_te_b)[:, 1]
results_b.append({'model': 'RandomForest',
                  'F1': f1_score(y_b_test, pred, average='weighted'),
                  'AUC': roc_auc_score(y_b_test, prob)})

# ── XGBoost ───────────────────────────────────────────────────────────────────
neg, pos = (y_b_train == 0).sum(), (y_b_train == 1).sum()
spw = neg / pos
xgb_clf = xgb.XGBClassifier(
    n_estimators=500, max_depth=5, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    scale_pos_weight=spw,
    random_state=42, n_jobs=-1, verbosity=0,
    eval_metric='aucpr', early_stopping_rounds=30
)
xgb_clf.fit(X_tr_b, y_b_train, eval_set=[(X_te_b, y_b_test)], verbose=False)
pred = xgb_clf.predict(X_te_b)
prob = xgb_clf.predict_proba(X_te_b)[:, 1]
results_b.append({'model': 'XGBoost',
                  'F1': f1_score(y_b_test, pred, average='weighted'),
                  'AUC': roc_auc_score(y_b_test, prob)})

print("\nTask B Summary (leakage fixed):")
print(pd.DataFrame(results_b).sort_values('AUC', ascending=False).to_string(index=False))

              precision    recall  f1-score   support

  No closure       1.00      1.00      1.00      1498
     Closure       0.99      1.00      1.00       137

    accuracy                           1.00      1635
   macro avg       1.00      1.00      1.00      1635
weighted avg       1.00      1.00      1.00      1635


Task B Summary (leakage fixed):
       model       F1      AUC
     XGBoost 0.999389 0.999854
RandomForest 0.999389 0.999615
 LogisticReg 0.999389 0.999454


In [3]:
# file4_stratified_models.py
# Technique: train separate XGBoost models per severity group
# Short incidents (< 30 min) have different drivers than critical ones (> 4 hrs)
# Combining them forces one model to fit two fundamentally different regimes

import pandas as pd
import numpy as np
import joblib, os, warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import KFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import xgboost as xgb
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

# ─── [Full preprocessing — paste from file1] ─────────────────────────────────
for col in ['closed_datetime','start_datetime','resolved_datetime','modified_datetime','created_date']:
    df[col] = pd.to_datetime(df[col], utc=True, errors='coerce')
df['end_ts']        = df['closed_datetime'].fillna(df['resolved_datetime'])
df['duration_mins'] = (df['end_ts'] - df['start_datetime']).dt.total_seconds() / 60
df['severity']     = pd.cut(df['duration_mins'], bins=[0,30,90,240,1440],
                             labels=['low','medium','high','critical'], include_lowest=True)
df['severity_enc'] = df['severity'].map({'low':0,'medium':1,'high':2,'critical':3})
mask = (df['duration_mins'].notna() & df['severity_enc'].notna() &
        (df['duration_mins']>0) & (df['duration_mins']<1440) &
        df['priority'].notna() & df['address'].notna())
df_clean = df[mask].copy()

for col in ['zone','junction','gba_identifier','veh_type','corridor','event_cause','event_type','status']:
    df_clean[col] = df_clean[col].fillna('Unknown')
df_clean['requires_road_closure'] = df_clean['requires_road_closure'].astype(int)

df_clean['hour']        = df_clean['start_datetime'].dt.hour
df_clean['day_of_week'] = df_clean['start_datetime'].dt.dayofweek
df_clean['month']       = df_clean['start_datetime'].dt.month
df_clean['is_weekend']  = df_clean['day_of_week'].isin([5,6]).astype(int)
df_clean['is_peak']     = df_clean['hour'].isin([7,8,9,17,18,19]).astype(int)
df_clean['is_night']    = df_clean['hour'].isin([22,23,0,1,2]).astype(int)
df_clean['hour_sin']    = np.sin(2*np.pi*df_clean['hour']/24)
df_clean['hour_cos']    = np.cos(2*np.pi*df_clean['hour']/24)
df_clean['dist_from_center'] = np.sqrt(
    (df_clean['latitude']-12.9716)**2+(df_clean['longitude']-77.5946)**2)
df_clean['has_end_location'] = (df_clean['endlatitude'].fillna(0)!=0).astype(int)
df_clean['lat_bin'] = pd.cut(df_clean['latitude'],  bins=10, labels=False)
df_clean['lon_bin'] = pd.cut(df_clean['longitude'], bins=10, labels=False)
df_clean['geo_span'] = np.sqrt(
    (df_clean['endlatitude'].fillna(df_clean['latitude'])-df_clean['latitude'])**2+
    (df_clean['endlongitude'].fillna(df_clean['longitude'])-df_clean['longitude'])**2)

LE_COLS = ['event_type','event_cause','veh_type','status']
le_encoders = {}
for col in LE_COLS:
    le = LabelEncoder()
    df_clean[f'{col}_enc'] = le.fit_transform(df_clean[col].astype(str))
    le_encoders[col] = le
df_clean['priority_enc'] = df_clean['priority'].str.lower().map(
    {'low':0,'medium':1,'high':2,'critical':3}).fillna(1).astype(int)

for col in ['zone','junction','corridor','gba_identifier','police_station']:
    if col in df_clean.columns:
        mean_map = df_clean.groupby(col)['duration_mins'].mean()
        df_clean[f'{col}_tenc'] = df_clean[col].map(mean_map).fillna(df_clean['duration_mins'].mean())
        df_clean[f'{col}_known'] = (df_clean[col] != 'Unknown').astype(int)

df_clean['cause_x_peak']    = df_clean['event_cause_enc'] * df_clean['is_peak']
df_clean['cause_x_closure'] = df_clean['event_cause_enc'] * df_clean['requires_road_closure']
df_clean['zone_x_peak']     = df_clean['zone_tenc'] * df_clean['is_peak']
df_clean['cause_hour_mean'] = df_clean.groupby(
    ['event_cause','hour'])['duration_mins'].transform('mean')
df_clean['zone_hour_mean']  = df_clean.groupby(
    ['zone','hour'])['duration_mins'].transform('mean')

FEATURE_COLS = [
    'hour','day_of_week','month','is_weekend','is_peak','is_night',
    'hour_sin','hour_cos',
    'latitude','longitude','lat_bin','lon_bin','has_end_location','geo_span','dist_from_center',
    'event_type_enc','event_cause_enc','requires_road_closure','veh_type_enc','status_enc','priority_enc',
    'zone_tenc','zone_known','junction_tenc','junction_known',
    'corridor_tenc','corridor_known','gba_identifier_tenc','gba_identifier_known',
    'cause_x_peak','cause_x_closure','zone_x_peak','cause_hour_mean','zone_hour_mean',
]
FEATURE_COLS = [c for c in FEATURE_COLS if c in df_clean.columns]
X     = df_clean[FEATURE_COLS].fillna(-999)
y_raw = df_clean['duration_mins']
y_log = np.log1p(y_raw)
kf    = KFold(n_splits=5, shuffle=True, random_state=42)

# ──────────────────────────────────────────────────────────────────────────────
#  STAGE 1: Train a classifier to predict severity bin
# ──────────────────────────────────────────────────────────────────────────────
from sklearn.ensemble import RandomForestClassifier

print("Stage 1: Training severity classifier...")
clf = RandomForestClassifier(
    n_estimators=300, max_depth=8,
    class_weight='balanced', random_state=42, n_jobs=-1
)
y_sev = df_clean['severity_enc'].astype(int)

oof_sev_pred = np.zeros(len(X), dtype=int)
for tr_idx, va_idx in kf.split(X):
    clf.fit(X.iloc[tr_idx], y_sev.iloc[tr_idx])
    oof_sev_pred[va_idx] = clf.predict(X.iloc[va_idx])

from sklearn.metrics import f1_score
print(f"Classifier OOF F1: {f1_score(y_sev, oof_sev_pred, average='weighted'):.3f}")

# ──────────────────────────────────────────────────────────────────────────────
#  STAGE 2: Train one XGBoost per severity group on log-duration
# ──────────────────────────────────────────────────────────────────────────────
SEVERITY_PARAMS = {
    # Low (< 30 min): fast resolution, low variance → shallower tree
    0: {'n_estimators':400, 'max_depth':4, 'learning_rate':0.05,
        'subsample':0.8, 'colsample_bytree':0.8, 'reg_alpha':0.5},
    # Medium (30-90 min): main bulk of data
    1: {'n_estimators':600, 'max_depth':6, 'learning_rate':0.05,
        'subsample':0.8, 'colsample_bytree':0.8, 'reg_alpha':0.5},
    # High (90-240 min): fewer samples, more regularization
    2: {'n_estimators':500, 'max_depth':5, 'learning_rate':0.03,
        'subsample':0.7, 'colsample_bytree':0.7, 'reg_alpha':1.0},
    # Critical (>240 min): fewest samples, heavy regularization
    3: {'n_estimators':300, 'max_depth':4, 'learning_rate':0.02,
        'subsample':0.6, 'colsample_bytree':0.6, 'reg_alpha':2.0},
}

# Optuna-tune the medium group (largest, most important)
print("\nOptuna-tuning medium severity group...")
mask_med = (y_sev == 1)
X_med, y_med = X[mask_med], y_log[mask_med]

def obj_med(trial):
    params = {
        'n_estimators':     trial.suggest_int('n_estimators', 200, 1000),
        'max_depth':        trial.suggest_int('max_depth', 3, 8),
        'learning_rate':    trial.suggest_float('learning_rate', 0.005, 0.15, log=True),
        'subsample':        trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'reg_alpha':        trial.suggest_float('reg_alpha', 0.0, 2.0),
        'reg_lambda':       trial.suggest_float('reg_lambda', 0.5, 4.0),
        'random_state': 42, 'verbosity': 0, 'n_jobs': -1,
    }
    kf2 = KFold(n_splits=3, shuffle=True, random_state=42)
    maes = []
    for tr, va in kf2.split(X_med):
        m = xgb.XGBRegressor(**params)
        m.fit(X_med.iloc[tr], y_med.iloc[tr])
        maes.append(mean_absolute_error(y_raw[mask_med].iloc[va],
                                        np.expm1(m.predict(X_med.iloc[va]))))
    return np.mean(maes)

study_med = optuna.create_study(direction='minimize',
                                 sampler=optuna.samplers.TPESampler(seed=42))
study_med.optimize(obj_med, n_trials=60, show_progress_bar=True)
SEVERITY_PARAMS[1] = study_med.best_params
print(f"Medium group best MAE: {study_med.best_value:.2f}")

# ──────────────────────────────────────────────────────────────────────────────
#  STAGE 3: OOF evaluation with routing
# ──────────────────────────────────────────────────────────────────────────────
print("\nOOF evaluation with stratified routing...")
oof_final = np.zeros(len(X))

group_models = {}  # will be refit later

for tr_idx, va_idx in kf.split(X):
    X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
    y_tr_raw, y_va_raw = y_raw.iloc[tr_idx], y_raw.iloc[va_idx]
    y_tr_sev = y_sev.iloc[tr_idx]

    # Train classifier on this fold
    fold_clf = RandomForestClassifier(
        n_estimators=200, max_depth=7, class_weight='balanced',
        random_state=42, n_jobs=-1
    )
    fold_clf.fit(X_tr, y_tr_sev)
    predicted_sev = fold_clf.predict(X_va)

    fold_preds = np.zeros(len(va_idx))

    for sev_group in [0, 1, 2, 3]:
        # Train group model on training rows truly in this group
        grp_tr_mask = (y_tr_sev == sev_group).values
        if grp_tr_mask.sum() < 10:
            # Too few — use global mean for this group
            fold_preds[predicted_sev == sev_group] = y_tr_raw[grp_tr_mask].mean() if grp_tr_mask.sum() > 0 else y_tr_raw.mean()
            continue

        grp_model = xgb.XGBRegressor(
            **SEVERITY_PARAMS[sev_group],
            objective='reg:squarederror',
            verbosity=0, n_jobs=-1, random_state=42
        )
        grp_model.fit(X_tr[grp_tr_mask], np.log1p(y_tr_raw[grp_tr_mask]))

        # Predict for validation rows routed to this group
        va_grp_mask = (predicted_sev == sev_group)
        if va_grp_mask.sum() > 0:
            fold_preds[va_grp_mask] = np.expm1(grp_model.predict(X_va[va_grp_mask]))

    oof_final[va_idx] = fold_preds

mae_strat  = mean_absolute_error(y_raw, oof_final)
rmse_strat = np.sqrt(mean_squared_error(y_raw, oof_final))
r2_strat   = r2_score(y_raw, oof_final)

print(f"\n{'='*45}")
print(f"  FILE 4 RESULT — Stratified per-severity models")
print(f"  OOF MAE:  {mae_strat:.2f} mins")
print(f"  OOF RMSE: {rmse_strat:.2f} mins")
print(f"  OOF R²:   {r2_strat:.4f}")
print(f"{'='*45}")

# ─── Save classifier + per-group models (refit on all data) ──────────────────
os.makedirs('models_v2', exist_ok=True)
clf.fit(X, y_sev)
joblib.dump(clf, 'models_v2/severity_router.pkl')

for sev_group in [0, 1, 2, 3]:
    grp_mask = (y_sev == sev_group).values
    if grp_mask.sum() < 10:
        continue
    gm = xgb.XGBRegressor(**SEVERITY_PARAMS[sev_group], verbosity=0, n_jobs=-1, random_state=42)
    gm.fit(X[grp_mask], y_log[grp_mask])
    joblib.dump(gm, f'models_v2/xgb_group_{sev_group}.pkl')

joblib.dump(FEATURE_COLS, 'models_v2/feature_cols_strat.pkl')
print("Saved → models_v2/ (router + 4 group models)")

Stage 1: Training severity classifier...
Classifier OOF F1: 0.421

Optuna-tuning medium severity group...


  0%|          | 0/60 [00:00<?, ?it/s]

Medium group best MAE: 14.63

OOF evaluation with stratified routing...

  FILE 4 RESULT — Stratified per-severity models
  OOF MAE:  104.32 mins
  OOF RMSE: 217.73 mins
  OOF R²:   -0.1305
Saved → models_v2/ (router + 4 group models)
